In [3]:
%load_ext autoreload
%autoreload 2


import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import polars as pl

sys.path.append('../')

from data_pipeline import load_all_raw_data


from plots.data_plot import (
    plot_user_analysis, plot_temporal_analysis, 
    plot_station_analysis, plot_activity_heatmap, 
    print_summary_statistics
)

from preprocess import (
    analyze_users_for_visualization, 
    analyze_trips_for_visualization
)

from weather_data import WeatherDataCollector


In [4]:
trips_with_weather = pl.read_parquet('../../data/processed/trips_with_weather.parquet')

trips_with_weather.head()

# trips_with_weather.columns


id_recorrido,duracion_recorrido,fecha_origen_recorrido,id_estacion_origen,nombre_estacion_origen,direccion_estacion_origen,long_estacion_origen,lat_estacion_origen,fecha_destino_recorrido,id_estacion_destino,nombre_estacion_destino,direccion_estacion_destino,long_estacion_destino,lat_estacion_destino,id_usuario,modelo_bicicleta,genero,weather_temperature_2m,weather_relative_humidity_2m,weather_dew_point_2m,weather_apparent_temperature,weather_precipitation,weather_rain,weather_weather_code,weather_pressure_msl,weather_surface_pressure,weather_cloud_cover,weather_cloud_cover_low,weather_cloud_cover_mid,weather_cloud_cover_high,weather_et0_fao_evapotranspiration,weather_vapour_pressure_deficit,weather_wind_speed_10m,weather_wind_speed_100m,weather_wind_direction_10m,weather_wind_direction_100m,weather_wind_gusts_10m,weather_soil_temperature_0_to_7cm,weather_soil_temperature_7_to_28cm,weather_soil_temperature_28_to_100cm,weather_soil_temperature_100_to_255cm,weather_soil_moisture_0_to_7cm,weather_soil_moisture_7_to_28cm,weather_soil_moisture_28_to_100cm,weather_soil_moisture_100_to_255cm,weather_sunshine_duration,weather_is_day,weather_direct_radiation
i64,i64,str,i64,str,str,f64,f64,str,i64,str,str,f64,f64,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
7210548,1582,"""2020-01-24 21:54:39""",27,"""027 - Montevideo""","""Cordoba Av. & Montevideo""",-58.390089,-34.599068,"""2020-01-24 22:21:01""",3,"""003 - ADUANA""","""Moreno & Av Paseo Colon""",-58.36826,-34.611032,192009,"""ICONIC""","""FEMALE""",30.311,53.307747,19.761,31.59533,0.0,0.0,1.0,1006.8,1004.6493,45.0,0.0,7.0,43.0,0.2730162,2.015435,16.808569,23.400002,350.13425,345.74994,33.839996,33.060997,27.761,23.811,20.211,0.179,0.354,0.401,0.452,3600.0,1.0,190.0
7199093,204,"""2020-01-24 07:04:19""",151,"""151 - AIME PAINÉ""","""Villaflor, Azucena & Paine, Ai…",-58.361285,-34.611815,"""2020-01-24 07:07:43""",3,"""003 - ADUANA""","""Moreno & Av Paseo Colon""",-58.36826,-34.611032,36380,"""ICONIC""","""MALE""",24.211,70.07739,18.411001,25.07353,0.0,0.0,2.0,1011.9,1009.6941,50.0,0.0,0.0,50.0,0.060814,0.903983,15.328561,27.193705,350.53775,353.15732,28.08,24.361,26.661001,23.761,20.211,0.197,0.36,0.401,0.452,0.0,0.0,0.0
7196805,1790,"""2020-01-24 00:15:17""",111,"""111 - MACACHA GUEMES""","""Machaca Guemes 350""",-58.364686,-34.605488,"""2020-01-24 00:45:07""",3,"""003 - ADUANA""","""Moreno & Av Paseo Colon""",-58.36826,-34.611032,460080,"""ICONIC""","""MALE""",24.761,79.67906,21.011,26.88475,0.0,0.0,0.0,1012.2,1009.9975,0.0,0.0,0.0,0.0,0.038332,0.6342735,15.496736,23.177402,87.33706,83.75819,25.199999,29.311,27.361,23.661001,20.161001,0.197,0.361,0.401,0.452,0.0,0.0,0.0
7203598,10688,"""2020-01-24 12:38:16""",285,"""400 - Reserva Ecologica""","""Achaval Rodriguez, T., Dr. Av.…",-58.356175,-34.617212,"""2020-01-24 15:36:24""",4,"""004 - Plaza Roma""","""Lavalle & Bouchard""",-58.368781,-34.601822,3857,"""ICONIC""","""MALE""",26.011,62.75473,18.361,26.85348,0.0,0.0,3.0,1012.5,1010.3061,91.0,0.0,0.0,91.0,0.43236,1.2523198,18.11841,22.59699,339.04413,337.5205,37.44,26.311,25.711,23.761,20.211,0.198,0.361,0.401,0.452,3600.0,1.0,498.0
7200335,673,"""2020-01-24 08:31:01""",171,"""171 - Pasteur""","""519 Pasteur""",-58.399755,-34.603281,"""2020-01-24 08:42:14""",7,"""007 - OBELISCO""","""CARLOS PELEGRINI 215""",-58.381098,-34.606498,391034,"""ICONIC""","""FEMALE""",23.161001,77.50442,19.011,24.020298,0.0,0.0,0.0,1012.1,1009.8857,0.0,0.0,0.0,0.0,0.04618,0.6380112,17.283749,28.08923,1.1934711,1.4687687,28.44,23.711,26.361,23.761,20.211,0.197,0.36,0.401,0.452,0.0,0.0,0.0


In [11]:
# trips_with_weather['fecha_origen_recorrido'] = pd.to_datetime(trips_with_weather['fecha_origen_recorrido'])
# trips_with_weather = trips_with_weather[trips_with_weather['fecha_origen_recorrido'].dt.year >= 2023]

# trips_with_weather.write_parquet('../../data/processed/trips_with_weather.parquet')

In [5]:

trips_with_weather.null_count()


id_recorrido,duracion_recorrido,fecha_origen_recorrido,id_estacion_origen,nombre_estacion_origen,direccion_estacion_origen,long_estacion_origen,lat_estacion_origen,fecha_destino_recorrido,id_estacion_destino,nombre_estacion_destino,direccion_estacion_destino,long_estacion_destino,lat_estacion_destino,id_usuario,modelo_bicicleta,genero,weather_temperature_2m,weather_relative_humidity_2m,weather_dew_point_2m,weather_apparent_temperature,weather_precipitation,weather_rain,weather_weather_code,weather_pressure_msl,weather_surface_pressure,weather_cloud_cover,weather_cloud_cover_low,weather_cloud_cover_mid,weather_cloud_cover_high,weather_et0_fao_evapotranspiration,weather_vapour_pressure_deficit,weather_wind_speed_10m,weather_wind_speed_100m,weather_wind_direction_10m,weather_wind_direction_100m,weather_wind_gusts_10m,weather_soil_temperature_0_to_7cm,weather_soil_temperature_7_to_28cm,weather_soil_temperature_28_to_100cm,weather_soil_temperature_100_to_255cm,weather_soil_moisture_0_to_7cm,weather_soil_moisture_7_to_28cm,weather_soil_moisture_28_to_100cm,weather_soil_moisture_100_to_255cm,weather_sunshine_duration,weather_is_day,weather_direct_radiation
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,41,41,41,41,41,0,0,39976,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [8]:
from data_analysis import engineer_ecobici_features

df_feat = engineer_ecobici_features(trips_with_weather)

df_feat.describe()

In [6]:
df_feat = pl.read_parquet(r"../../data/processed/trips_feat_eng.parquet")



In [7]:
# Obtener solo las columnas que tienen nulls (> 0)
df_nulls = df_feat.null_count()

null_summary = df_nulls.transpose(include_header=True)
null_summary.columns = ["column", "null_count"]

cols_with_nulls = null_summary.filter(pl.col("null_count") > 0)
print(cols_with_nulls)


shape: (9, 2)
┌───────────────────────┬────────────┐
│ column                ┆ null_count │
│ ---                   ┆ ---        │
│ str                   ┆ u32        │
╞═══════════════════════╪════════════╡
│ trip_dur_mean_last_DT ┆ 8448894    │
│ share_male            ┆ 8448894    │
│ share_female          ┆ 8448894    │
│ share_other           ┆ 8448894    │
│ precip_flag           ┆ 22860      │
│ wind_dir_sin          ┆ 22860      │
│ wind_dir_cos          ┆ 22860      │
│ precipitation_lag_1h  ┆ 23241      │
│ rain_lag_1h           ┆ 23241      │
└───────────────────────┴────────────┘


In [8]:
df_feat.head()

station_id,ts_start,dep_last_DT,trip_dur_mean_last_DT,model_FIT_cnt,model_ICONIC_cnt,share_male,share_female,share_other,dep_lag_1,dep_lag_2,dep_lag_3,dep_lag_4,dep_lag_5,dep_lag_6,arr_last_DT,arr_lag_1,arr_lag_2,arr_lag_3,arr_lag_4,arr_lag_5,arr_lag_6,y_arrivals_next_DT,weather_temperature_2m,weather_relative_humidity_2m,weather_dew_point_2m,weather_apparent_temperature,weather_precipitation,weather_rain,weather_weather_code,weather_pressure_msl,weather_surface_pressure,weather_cloud_cover,weather_cloud_cover_low,weather_cloud_cover_mid,weather_cloud_cover_high,weather_et0_fao_evapotranspiration,…,weather_soil_temperature_0_to_7cm,weather_soil_temperature_7_to_28cm,weather_soil_temperature_28_to_100cm,weather_soil_temperature_100_to_255cm,weather_soil_moisture_0_to_7cm,weather_soil_moisture_7_to_28cm,weather_soil_moisture_28_to_100cm,weather_soil_moisture_100_to_255cm,weather_sunshine_duration,weather_is_day,weather_direct_radiation,precip_flag,wind_dir_sin,wind_dir_cos,weather_code_cat_Clear,weather_code_cat_Clouds,weather_code_cat_Drizzle,weather_code_cat_Rain,weather_code_cat_null,precipitation_lag_1h,rain_lag_1h,is_holiday_ar,sin_hour,cos_hour,sin_dow,cos_dow,sin_month,cos_month,is_weekend,payday_flag,vacation_season,peak_commute,dep_ma_24h,dep_std_24h,dep_ratio_DT_24h,near_dep_sum_DT,near_dep_lag_1
i64,datetime[μs],u32,f64,u32,u32,f64,f64,f64,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,f64,f64,u8,u8,u8,u8,u8,f64,f64,i8,f32,f32,f32,f32,f32,f32,i8,i8,i8,i8,f32,f32,f32,u32,u32
2,2023-01-01 00:00:00,0,null,0,0,null,null,null,0,0,0,0,0,0,0,0,0,0,0,0,0,0,22.261,69.92862,16.511,19.90884,0.0,0.0,3.0,1012.3,1010.0787,100.0,0.0,0.0,96.0,0.124625,…,27.361,27.611,23.561,19.511,0.157,0.321,0.426,0.43,0.0,0.0,0.0,0,0.931577,-0.363544,0,1,0,0,0,null,null,1,0.0,1.0,-2.4493e-16,1.0,0.5,0.866025,0,1,1,0,0.0,0.0,0.0,0,0
3,2023-01-01 00:00:00,0,null,0,0,null,null,null,0,0,0,0,0,0,0,0,0,0,0,0,0,1,22.261,69.92862,16.511,19.90884,0.0,0.0,3.0,1012.3,1010.0787,100.0,0.0,0.0,96.0,0.124625,…,27.361,27.611,23.561,19.511,0.157,0.321,0.426,0.43,0.0,0.0,0.0,0,0.931577,-0.363544,0,1,0,0,0,null,null,1,0.0,1.0,-2.4493e-16,1.0,0.5,0.866025,0,1,1,0,0.0,0.0,0.0,0,0
4,2023-01-01 00:00:00,0,null,0,0,null,null,null,0,0,0,0,0,0,0,0,0,0,0,0,0,0,22.261,69.92862,16.511,19.90884,0.0,0.0,3.0,1012.3,1010.0787,100.0,0.0,0.0,96.0,0.124625,…,27.361,27.611,23.561,19.511,0.157,0.321,0.426,0.43,0.0,0.0,0.0,0,0.931577,-0.363544,0,1,0,0,0,null,null,1,0.0,1.0,-2.4493e-16,1.0,0.5,0.866025,0,1,1,0,0.0,0.0,0.0,1,0
5,2023-01-01 00:00:00,2,872.5,2,0,0.5,0.0,0.5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,22.261,69.92862,16.511,19.90884,0.0,0.0,3.0,1012.3,1010.0787,100.0,0.0,0.0,96.0,0.124625,…,27.361,27.611,23.561,19.511,0.157,0.321,0.426,0.43,0.0,0.0,0.0,0,0.931577,-0.363544,0,1,0,0,0,null,null,1,0.0,1.0,-2.4493e-16,1.0,0.5,0.866025,0,1,1,0,2.0,0.0,0.666667,0,0
6,2023-01-01 00:00:00,0,null,0,0,null,null,null,0,0,0,0,0,0,0,0,0,0,0,0,0,0,22.261,69.92862,16.511,19.90884,0.0,0.0,3.0,1012.3,1010.0787,100.0,0.0,0.0,96.0,0.124625,…,27.361,27.611,23.561,19.511,0.157,0.321,0.426,0.43,0.0,0.0,0.0,0,0.931577,-0.363544,0,1,0,0,0,null,null,1,0.0,1.0,-2.4493e-16,1.0,0.5,0.866025,0,1,1,0,0.0,0.0,0.0,0,0


In [9]:
code_cols = ["ts_start"]

target_col = 'y_arrivals_next_DT'

feat_cols = ['station_id', 'dep_last_DT', 'trip_dur_mean_last_DT',
       'model_FIT_cnt', 'model_ICONIC_cnt', 'share_male', 'share_female',
       'share_other', 'dep_lag_1', 'dep_lag_2', 'dep_lag_3', 'dep_lag_4',
       'dep_lag_5', 'dep_lag_6', 'arr_last_DT', 'arr_lag_1', 'arr_lag_2',
       'arr_lag_3', 'arr_lag_4', 'arr_lag_5', 'arr_lag_6', 'weather_temperature_2m',
       'weather_relative_humidity_2m', 'weather_dew_point_2m',
       'weather_apparent_temperature', 'weather_precipitation', 'weather_rain',
       'weather_pressure_msl', 'weather_surface_pressure', 'weather_cloud_cover',
       'weather_cloud_cover_low', 'weather_cloud_cover_mid',
       'weather_cloud_cover_high', 'weather_et0_fao_evapotranspiration',
       'weather_vapour_pressure_deficit',
       'weather_soil_temperature_0_to_7cm',
       'weather_soil_moisture_0_to_7cm', 'weather_sunshine_duration',
       'weather_is_day', 'weather_direct_radiation', 'precip_flag',
       'wind_dir_sin', 'wind_dir_cos', 'weather_code_cat_Clear',
       'weather_code_cat_Clouds', 'weather_code_cat_Drizzle',
       'weather_code_cat_Rain', 'weather_code_cat_null',
       'precipitation_lag_1h', 'rain_lag_1h', 'is_holiday_ar', 'sin_hour',
       'cos_hour', 'sin_dow', 'cos_dow', 'sin_month', 'cos_month',
       'is_weekend', 'payday_flag', 'vacation_season', 'peak_commute',
       'dep_ma_24h', 'dep_std_24h', 'dep_ratio_DT_24h', 'near_dep_sum_DT',
       'near_dep_lag_1']



In [10]:
df_feat = df_feat[code_cols + feat_cols + [target_col]]

QUEDA DROPPEAR ALGUNAS COLUMNAS INNECESARIAS DE WEATHER Y ESTAMOS!

In [9]:
df_feat.sample(5)


#count number of uniques ids 

df_feat.select(pl.col("station_id").unique().count())

station_id
u32
381


In [14]:
"ts_start" in df_feat.columns

True

In [11]:
from data_analysis import pivot_features_for_global_model

df_feat_pivot = pivot_features_for_global_model(df_feat)



Pivoting data to wide format for global model...

[Step 1/3] Identifying columns...

[Step 2/3] Constructing feature (X) matrix...
  -> Pivoting station-specific features...
  -> Saving checkpoint: ../../data/processed/feature_checkpoints/X_station_wide.parquet
  -> Extracting global features...
  -> Joining global and station-specific features...
  -> Saving final features matrix: ../../data/processed/X_wide.parquet

[Step 3/3] Constructing target (y) matrix...
  -> Pivoting target variable...
  -> Saving checkpoint: ../../data/processed/feature_checkpoints/y_wide_unaligned.parquet
  -> Aligning target matrix with feature matrix timestamps...
  -> Saving final target matrix: ../../data/processed/y_wide.parquet
--------------------------------------------------
  Original long shape: (11118723, 68)
  Pivoted X_wide shape: (29183, 9566) -> saved to X_wide.parquet
  Pivoted y_wide shape: (29183, 382) -> saved to y_wide.parquet
Pivoting complete.


In [11]:
X_wide = pl.read_parquet(r"../../data/processed/X_wide.parquet")

In [12]:
X_wide.shape

(29183, 9566)

In [14]:
X_wide.head()

ts_start,weather_temperature_2m,weather_relative_humidity_2m,weather_dew_point_2m,weather_apparent_temperature,weather_precipitation,weather_rain,weather_pressure_msl,weather_surface_pressure,weather_cloud_cover,weather_cloud_cover_low,weather_cloud_cover_mid,weather_cloud_cover_high,weather_et0_fao_evapotranspiration,weather_vapour_pressure_deficit,weather_soil_temperature_0_to_7cm,weather_soil_moisture_0_to_7cm,weather_sunshine_duration,weather_is_day,weather_direct_radiation,precip_flag,wind_dir_sin,wind_dir_cos,weather_code_cat_Clear,weather_code_cat_Clouds,weather_code_cat_Drizzle,weather_code_cat_Rain,weather_code_cat_null,precipitation_lag_1h,rain_lag_1h,is_holiday_ar,sin_hour,cos_hour,sin_dow,cos_dow,sin_month,cos_month,…,near_dep_lag_1_519,near_dep_lag_1_520,near_dep_lag_1_521,near_dep_lag_1_522,near_dep_lag_1_523,near_dep_lag_1_524,near_dep_lag_1_525,near_dep_lag_1_526,near_dep_lag_1_527,near_dep_lag_1_528,near_dep_lag_1_529,near_dep_lag_1_530,near_dep_lag_1_531,near_dep_lag_1_532,near_dep_lag_1_533,near_dep_lag_1_534,near_dep_lag_1_535,near_dep_lag_1_536,near_dep_lag_1_537,near_dep_lag_1_538,near_dep_lag_1_539,near_dep_lag_1_540,near_dep_lag_1_541,near_dep_lag_1_542,near_dep_lag_1_543,near_dep_lag_1_544,near_dep_lag_1_545,near_dep_lag_1_546,near_dep_lag_1_547,near_dep_lag_1_548,near_dep_lag_1_549,near_dep_lag_1_550,near_dep_lag_1_551,near_dep_lag_1_552,near_dep_lag_1_553,near_dep_lag_1_554,near_dep_lag_1_556
datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,f64,f64,u8,u8,u8,u8,u8,f64,f64,i8,f32,f32,f32,f32,f32,f32,…,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
2023-01-01 00:00:00,22.261,69.92862,16.511,19.90884,0.0,0.0,1012.3,1010.0787,100.0,0.0,0.0,96.0,0.124625,0.807796,27.361,0.157,0.0,0.0,0.0,0,0.931577,-0.363544,0,1,0,0,0,null,null,1,0.0,1.0,-2.4493e-16,1.0,0.5,0.866025,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2023-01-01 00:30:00,22.261,69.92862,16.511,19.963741,0.0,0.0,1012.7,1010.4778,100.0,0.0,1.0,100.0,0.105228,0.807796,26.211,0.157,0.0,0.0,0.0,0,0.954275,-0.298931,0,1,0,0,0,0.0,0.0,1,0.0,1.0,-2.4493e-16,1.0,0.5,0.866025,…,0,0,0,0,0,0,0,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,1,2
2023-01-01 01:00:00,22.261,69.92862,16.511,19.963741,0.0,0.0,1012.7,1010.4778,100.0,0.0,1.0,100.0,0.105228,0.807796,26.211,0.157,0.0,0.0,0.0,0,0.954275,-0.298931,0,1,0,0,0,0.0,0.0,1,0.258819,0.965926,-2.4493e-16,1.0,0.5,0.866025,…,0,0,0,0,1,3,1,1,0,0,1,1,0,1,1,1,2,0,1,0,0,0,0,0,4,2,1,0,3,0,0,0,0,0,1,1,2
2023-01-01 01:30:00,22.411001,69.293495,16.511,20.185112,0.0,0.0,1012.4,1010.1797,100.0,0.0,3.0,100.0,0.107084,0.832404,25.461,0.157,0.0,0.0,0.0,0,0.969451,-0.245283,0,1,0,0,0,0.0,0.0,1,0.258819,0.965926,-2.4493e-16,1.0,0.5,0.866025,…,1,0,3,2,2,2,0,0,1,0,4,4,2,0,2,2,0,2,2,1,0,0,1,0,5,2,1,0,4,0,0,2,0,2,1,1,1
2023-01-01 02:00:00,22.411001,69.293495,16.511,20.185112,0.0,0.0,1012.4,1010.1797,100.0,0.0,3.0,100.0,0.107084,0.832404,25.461,0.157,0.0,0.0,0.0,0,0.969451,-0.245283,0,1,0,0,0,0.0,0.0,1,0.5,0.866025,-2.4493e-16,1.0,0.5,0.866025,…,0,0,0,5,1,3,0,0,0,0,6,3,0,0,1,4,2,1,1,1,2,2,2,0,11,1,0,0,3,0,0,0,0,2,0,1,1


In [15]:
y_wide = pl.read_parquet(r"../../data/processed/y_wide.parquet")

In [16]:
y_wide.shape

(29183, 382)

In [17]:


def check_nans(df: pl.DataFrame, name: str = "df") -> None:
    # Handle datetime columns separately from numeric columns
    datetime_cols = df.select(pl.col(pl.Datetime)).columns
    numeric_cols = [col for col in df.columns if col not in datetime_cols]
    
    # Get NaN counts for numeric columns
    per_col_numeric = (df.select([pl.col(col).is_nan().sum() for col in numeric_cols])
                      .row(0) if numeric_cols else [])
    
    # Get null counts for datetime columns 
    per_col_datetime = (df.select([pl.col(col).is_null().sum() for col in datetime_cols])
                       .row(0) if datetime_cols else [])
    
    # Combine results
    per_col = list(per_col_numeric) + list(per_col_datetime)
    total_nans = sum(per_col)

    if total_nans == 0:
        print(f"✅ {name}: no NaN/null values found.")
    else:
        print(f"⚠️  {name}: {total_nans:,} NaN/null values detected.")
        # Create DataFrame of columns with NaNs
        nan_cols = pl.DataFrame({
            "column": df.columns,
            "nans": per_col
        })
        (nan_cols
         .filter(pl.col("nans") > 0)
         .sort("nans", descending=True)
         .show())

# ------------------------------------------------------------------
check_nans(X_wide, "X_wide") 
check_nans(y_wide, "y_wide")

✅ X_wide: no NaN/null values found.
✅ y_wide: no NaN/null values found.
